# vorflow basic example

A minimal end-to-end run: a square domain with a refined zone, a river, a fault
barrier and a pumping well, turned into a Voronoi grid for MODFLOW 6.

The pipeline always runs the same three classes in order:

1. `ConceptualMesh` collects the features and cleans them (overlaps, snapping, clipping, densification).
2. `MeshGenerator` builds a triangular mesh with Gmsh.
3. `VoronoiTessellator` takes the dual of that mesh and returns the cells as a GeoDataFrame.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from shapely.geometry import LineString, Point, box

from vorflow import ConceptualMesh, MeshGenerator, VoronoiTessellator
from vorflow.utils import calculate_mesh_quality, summarize_quality

## 1. Geometry

Plain Shapely objects. Coordinates are in model units (here metres).

In [ ]:
domain = box(0, 0, 200, 200)
refined_zone = box(50, 50, 75, 75)
river = LineString([(x, 75 + 20 * np.sin(0.1 * x)) for x in range(-10, 220, 10)])
fault = LineString([(100, 0), (100, 150)])
well = Point(25, 25)

## 2. Conceptual model

- `resolution` is the target cell size on a feature.
- Away from a feature finer than `background_lc`, the size grows geometrically
  (`growth_factor`, default 1.2) until it reaches `background_lc`.
- `z_order` decides which polygon wins where polygons overlap.
- `is_barrier=True` lines cell faces up along the line, so no face crosses it and a
  horizontal flow barrier can be placed on the faces that lie on it.

The river starts and ends outside the domain; `generate()` clips it to the domain.

In [ ]:
background_lc = 20.0

blueprint = ConceptualMesh(crs="EPSG:3857")
blueprint.add_polygon(domain, zone_id=1)
blueprint.add_polygon(refined_zone, zone_id=2, resolution=2.0, z_order=1)
blueprint.add_line(river, line_id="river", resolution=2.0)
blueprint.add_line(fault, line_id="fault", resolution=2.0, is_barrier=True)
blueprint.add_point(well, point_id="well-a", resolution=1.0, growth_factor=1.3)

clean_polys, clean_lines, clean_pts = blueprint.generate()

## 3. Mesh and tessellate

`MeshGenerator.generate()` returns `True`. The mesh itself is stored on the
generator (`.nodes`, `.zones_gdf`, ...), and the tessellator reads it from there.

In [ ]:
mesher = MeshGenerator(background_lc=background_lc)
mesher.generate(clean_polys, clean_lines, clean_pts)

tessellator = VoronoiTessellator(mesher, blueprint, clip_to_boundary=True)
grid = tessellator.generate()
print(f"{len(grid)} cells, {tessellator.n_barrier_mirrors} barrier mirror cells")

The grid has one row per cell:

- `node_id`: unique cell ID.
- `x`, `y`: the **generator point**, i.e. the Voronoi site the cell was built around.
- `centroid_x`, `centroid_y`: the geometric centroid.
- `zone_id`, `z_order`: taken from the polygon that contains the generator.

In [ ]:
grid.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
grid.plot(ax=ax, column="zone_id", cmap="Pastel1", edgecolor="k", linewidth=0.2)
ax.plot(*river.xy, color="tab:blue", lw=1.5, label="River")
ax.plot(*fault.xy, color="tab:red", lw=1.5, label="Fault (barrier)")
ax.scatter(*well.xy, color="k", marker="x", s=40, label="Well", zorder=5)
ax.set_xlim(-5, 205)
ax.set_ylim(-5, 205)
ax.set_aspect("equal")
ax.legend(loc="upper left")
ax.set_title("Voronoi grid coloured by zone")
plt.show()

## 4. Mesh quality

`calculate_mesh_quality` adds per-cell metrics. `drift_ratio` is the distance
between generator and centroid divided by √area (0 is ideal).
`summarize_quality` reports internal and boundary cells separately.

In [ ]:
quality = calculate_mesh_quality(grid, calc_ortho=True)
summarize_quality(quality)

fig, ax = plt.subplots(figsize=(10, 8))
quality.plot(ax=ax, column="drift_ratio", cmap="viridis", legend=True, vmax=0.25)
ax.set_aspect("equal")
ax.set_title("Drift ratio")
plt.show()

## 5. Export

Writing a file needs a GeoPandas I/O engine such as Pyogrio or Fiona:

```python
grid.to_file("mf6_grid.gpkg", driver="GPKG")
```